In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from matplotlib import rc
import time
import random


## Question 1

### Question 1a

In [ ]:
# import polblog data
G = nx.DiGraph()
with open('data/polblogs_nodes_class.tsv','r') as fp:
    for line in fp:
        node_id, node_label, node_political = line.strip().split('\t')
        political_orientation = 'liberal' if node_political == '0' else 'conservative'
        G.add_node(node_id, website=node_label, political_orientation=political_orientation)

with open('data/polblogs_edges_class.tsv','r') as fp:
    for line in fp:
        source_node_id, target_node_id = line.strip().split('\t')
        G.add_edge(source_node_id, target_node_id)

In [ ]:
print("Number of nodes: %d" % G.number_of_nodes())
print("Number of edges: %d" % G.number_of_edges())
print("Graph density:\t %1.7f" % nx.density(G))

In [ ]:

# Write all the edgelists
import json
import pickle

# Edgelist
nx.write_edgelist(G, "outputs/polblogs.edgelist")

# GraphML
nx.write_graphml(G, "outputs/polblogs.graphml")

#JSON
json_graph = nx.node_link_data(G)
with open("outputs/polblogs.json", "w", encoding="utf-8") as gfile:
    json.dump(json_graph, gfile)

# Pickle binary serialization
with open("outputs/polblogs.pkl", 'wb') as gfile:
    pickle.dump(G, gfile)

# dense adjacency matrix
nmp = nx.to_numpy_array(G)
with open("outputs/polblogs.npy", 'wb') as gfile:
    np.save(gfile, nmp)


In [ ]:
# Read in data
import os

# First, write the original network's information
# Now save everything to a file for easy viewing 
with open("outputs/network_info.txt", 'w') as info_file:
    info_file.write(f"Network: G \n")
    info_file.write(f"Node Count: {G.number_of_nodes()}\n")
    info_file.write(f"Edge Count: {G.number_of_edges()}\n")
    info_file.write(f"Is directed: {nx.is_directed(G)}\n")
    info_file.write(f"Website Attr Length: {len(nx.get_node_attributes(G, 'website'))}\n")
    info_file.write(f"Political Orientation Attr Length: {len(nx.get_node_attributes(G, 'political_orientation'))}\n")
    info_file.write(f"Graphs are equal: {nx.utils.graphs_equal(G, G)}\n") 
    info_file.write(f"\n\n")

network_list = list()
suffixes = ["edgelist", "graphml", "json", "pkl", "npy"]

# Edgelist
network_list.append(nx.read_edgelist("outputs/polblogs.edgelist"))

# GraphML
network_list.append(nx.read_graphml("outputs/polblogs.graphml"))

# JSON
with open('outputs/polblogs.json', 'r', encoding='utf-8') as file:
    node_link_data = json.load(file)
network_list.append(nx.node_link_graph(node_link_data))

# Pickle binary serialization
with open("outputs/polblogs.pkl", 'rb') as gfile:
    network_list.append(pickle.load(gfile))

# NP dense ajacency matrix
with open("outputs/polblogs.npy", 'rb') as gfile:
    back_np = np.load(gfile)
    network_list.append(nx.from_numpy_array(back_np))

# Now save everything to the file for easy viewing 
with open("outputs/network_info.txt", 'a') as info_file:
    for i, network in enumerate(network_list):
        info_file.write(f"Network: {suffixes[i]} \n")
        info_file.write(f"File size: {round(os.path.getsize(f'outputs/polblogs.{suffixes[i]}') / 1024)} kilobytes \n")
        info_file.write(f"Node Count: {network.number_of_nodes()}\n")
        info_file.write(f"Edge Count: {network.number_of_edges()}\n")
        info_file.write(f"Is directed: {nx.is_directed(network)}\n")
        info_file.write(f"Website Attr Length: {len(nx.get_node_attributes(network, 'website'))}\n")
        info_file.write(f"Political Orientation Attr Length: {len(nx.get_node_attributes(network, 'political_orientation'))}\n")
        info_file.write(f"Graphs are equal: {nx.utils.graphs_equal(G, network)}\n")
        info_file.write("\n\n")

### Question 1b

In [ ]:
# Now check the specifics of the attributes 

# First, what is G?
print(f"Looking at network G")
attrs = nx.get_node_attributes(G, 'website')
print(f'Attribute storage type is {type(attrs)}')
print(f'Attribute type is {type(attrs["0"])}')
print(f"G nodes: {type(G.nodes)}")
print(("\n\n"))

# Now compare the others
for i, network in enumerate(network_list):
    attrs = nx.get_node_attributes(network, 'website')

    if(len(attrs) > 0):
        print(f"Looking at network {suffixes[i]}")
        print(f'Attribute storage type is {type(attrs)}')
        print(f'Attribute type is {type(attrs["0"])}')
        print(("\n"))

# Check np file
nparray = np.load('outputs/polblogs.npy')
nparray

### Question 1c

In [ ]:
# Fix Edgelist
nx.write_edgelist(G, "outputs/polblogs_fixed.edgelist", data=["website", "political_orientation"],  delimiter=',')
edge_list = nx.read_edgelist("outputs/polblogs_fixed.edgelist", create_using=nx.DiGraph,  delimiter=',')

print(f"Network: edgelist")
print(f"File size: {round(os.path.getsize(f'outputs/polblogs_fixed.edgelist') / 1024)} kilobytes")
print(f"Node Count: {edge_list.number_of_nodes()}")
print(f"Edge Count: {edge_list.number_of_edges()}")
print(f"Is directed: {nx.is_directed(edge_list)}")
print(f"Website Attr Length: {len(nx.get_node_attributes(edge_list, 'website'))}")
print(f"Political Orientation Attr Length: {len(nx.get_node_attributes(edge_list, 'political_orientation'))}")
print(f"Graphs are equal: {nx.utils.graphs_equal(G, edge_list)}\n")

In [ ]:
# Fix GraphML
nx.write_graphml(G, "outputs/polblogs_fixed.graphml")
GraphML_fixed = nx.read_graphml("outputs/polblogs_fixed.graphml")

print(f"Network: GraphML")
print(f"File size: {round(os.path.getsize(f'outputs/polblogs_fixed.graphml') / 1024)} kilobytes ")
print(f"Node Count: {GraphML_fixed.number_of_nodes()}")
print(f"Edge Count: {GraphML_fixed.number_of_edges()}")
print(f"Is directed: {nx.is_directed(GraphML_fixed)}")
print(f"Website Attr Length: {len(nx.get_node_attributes(GraphML_fixed, 'website'))}")
print(f"Political Orientation Attr Length: {len(nx.get_node_attributes(GraphML_fixed, 'political_orientation'))}")
print(f"Graphs are equal: {nx.utils.graphs_equal(G, GraphML_fixed)}\n")

# look for the differences
print(GraphML_fixed.nodes == G.nodes)
print(GraphML_fixed.edges == G.edges)
print(GraphML_fixed.edges == G.edges)
print(nx.get_node_attributes(GraphML_fixed, 'political_orientation')==nx.get_node_attributes(G, 'political_orientation'))
print(nx.get_node_attributes(GraphML_fixed, 'website')==nx.get_node_attributes(G, 'website'))

print("Looking at nodes and data....")
print(G.nodes(data=True) == GraphML_fixed.nodes(data=True))
#print(f"Same shape: {nx.is_isomorphic(G, GraphML_fixed)}")
print(type(G))
print(type(GraphML_fixed))


# Check IDs
node_ids = list(G)
graph_node_ids = list(GraphML_fixed)
print(graph_node_ids == node_ids)

# Everything checks out. Maybe some oddness with the python object itself. 
# Now check based on docstring
print("Looking at docstring evaluations...")
print(GraphML_fixed.adj == G.adj)
print(GraphML_fixed.nodes == G.nodes)
print(GraphML_fixed.graph == G.graph)
print("Done.")

# The graphs are not equal, let's print both
print(f"GraphM:: {GraphML_fixed.graph}")
print(f"Original: {G.graph}")

# The original is just an empty list. So reset 
GraphML_fixed.graph = {}
print("Updated new graph. Equal?")
print(GraphML_fixed.adj == G.adj)
print(GraphML_fixed.nodes == G.nodes)
print(GraphML_fixed.graph == G.graph)
print(nx.utils.graphs_equal(GraphML_fixed, G))
# Fixed it!

In [ ]:
# dense adjacency matrix
nmp = nx.to_numpy_array(G)
with open("outputs/polblogs_fixed.npy", 'wb') as gfile:
    np.save(gfile, nmp)

# NP dense ajacency matrix
# Pickle binary serialization
with open("outputs/polblogs_fixed.npy", 'rb') as gfile:
    back_np = np.load(gfile)
    np_network = nx.from_numpy_array(back_np, create_using=nx.DiGraph)

print(f"\n\nNetwork: NP Array")
print(f"File size: {round(os.path.getsize(f'outputs/polblogs_fixed.edgelist') / 1024)} kilobytes")
print(f"Node Count: {np_network.number_of_nodes()}")
print(f"Edge Count: {np_network.number_of_edges()}")
print(f"Is directed: {nx.is_directed(np_network)}")
print(f"Website Attr Length: {len(nx.get_node_attributes(np_network, 'website'))}")
print(f"Political Orientation Attr Length: {len(nx.get_node_attributes(np_network, 'political_orientation'))}")
print(f"Graphs are equal: {nx.utils.graphs_equal(G, np_network)}\n")

### Question 1e

In [ ]:
baby_graph = nx.path_graph(5)
print(baby_graph.nodes(data=True))


In [ ]:
# Edgelist
nx.write_edgelist(G, "outputs/baby_graph.edgelist")
baby_edge = nx.read_edgelist("outputs/baby_graph.edgelist")

# commenting out to save endless output
#baby_edge.nodes(data=True)

# labels stored as strings

In [ ]:
# GraphML
nx.write_graphml(G, "outputs/baby_graph.graphml")
baby_graphml = nx.read_graphml("outputs/baby_graph.graphml")

# commenting out to save endless output
#baby_graphml.nodes(data=True)

# labels also stored as strings

In [ ]:
# now try with int
baby_edge = nx.read_edgelist("outputs/baby_graph.edgelist", nodetype=int)
print(baby_edge.nodes(data=True))

baby_graphml = nx.read_graphml("outputs/baby_graph.graphml",  node_type=int)

# commenting out to save endless output
#print(baby_graphml.nodes(data=True))

# All fixed! 


### Question 1f

In [ ]:
# First with the write edge method of storing 

# size of current network
node_count = len(G.nodes)
write_link_count = 16718 # (its different because no edgeless nodes)

# I anticipate that edgewrite scales at O(N) except where N equals links
# So what proportion of links to nodes?
potential_links = node_count ** 2 # 2220100
link_proportion = write_link_count / potential_links # 0.00856943380928787 or .86%
# so number of links on a 10**6 network would be
big_potential_links = (10 ** 6 )**2
# and then we want .68% of that so
big_link_count = big_potential_links * link_proportion
# and now get kb size in proportion 
# link_count / file size = big_link_count / file size)
new_size = (big_link_count * 214)/write_link_count
print(new_size)

In [ ]:
# Now repeat with adjacency matrix 

# size of current network
node_count = len(G.nodes)

# I anticipate that adjacency network scales at O(N^2) because all links are represented
# So we don't need to deal with link count, just N
big_node_count = 10 ** 6

# so we have big_N^2/file_size = N^2/file_size
# so (BigN^2 * file_size) / N^2
new_size = (big_link_count ** 2 * 17345)/node_count ** 2

print(new_size)

## Question 3

### Question 3.1

In [ ]:
# Developing my own shortest path algorithm
# Because theirs was super slow
# Mine probably will be too
# But we'll try it out

# The recursion portion
# This basically takes in a list of predecessors of a given node, the current path 

def pred_recurse(pred, curr_path, r_source, all_paths, all_predecessors):
    """
    The recursion portion. This builds out all possibly shortest paths recursively
    based on predecessor nodes.

    @param pred (int): The predecessor node we are building off of
    @param curr_path (list): the current path as it is built
    @param source (int): the source node we are getting all shortest paths to
    @param all_paths (list of lists): A list of all paths we have accumulated for that source

    @return None because we are just adding to the list as we go
            That might be bad practice. Not sure.
    """
    new_path = [pred] + curr_path
    if new_path[0] == r_source:
        all_paths.append(new_path)
        return
    new_preds = all_predecessors[pred]
    for new_pred in new_preds: 
        pred_recurse(new_pred, new_path, r_source, all_paths, all_predecessors)



def run_pred_recurse_for_source(graph, source_node):
    """
    This runs the recursion portion for each 

    @param graph (object): The graph we are working with
    @param source_node (int): the source node we want all shortest paths from

    @return all_paths (list of lists): a list of all the shortest paths from the source to every other node
                                        Some node pairs have multiple shortest paths 
    """
    predecessors = nx.predecessor(graph, source_node) # a dictionary of predecessors for the path from source to all nodes in the graph.
    all_paths = [] 
    for init_pred in predecessors:
        pred_recurse(init_pred, [], source_node, all_paths, predecessors)   
    return all_paths



def test_my_shortest_paths(graph, test_source):
    """
    This just tests my shortest graph algorithm against Network X's. 

    @param graph (object): The graph we are working with
    @param source_node (int): the source node we want to test shortest paths from

    @return success (bool), string of results
    """
    my_paths = run_pred_recurse_for_source(graph, test_source)
    incorrect_nodes = []
    for node in graph.nodes:
        my_path_list = [my_p for my_p in my_paths if my_p[-1] == node]
        their_path_list = [p for p in nx.all_shortest_paths(graph, test_source, node)] 
        if (not my_path_list == their_path_list):
            incorrect_nodes.append(node)

    if(len(incorrect_nodes) > 0):
        return False, f"Found the following target nodes for source {test_source} with mismatched paths: {incorrect_nodes}"
    else:
        return True, f"All paths are equal to Network X's paths for source {test_source}!"

In [ ]:
# Now we test our function against NetworkX's function for all the karate nodes
K = nx.karate_club_graph()

final_result = True
# Lets test for all nodes as source nodes
for test_node in K.nodes:
    result, er = test_my_shortest_paths(K, test_node)
    final_result = result
    if result == False:
        print(er) 

if final_result:
    print("All tests passed!")
# Everything looks good 


In [ ]:
# Now test time - who is faster?

start_time = time.perf_counter()
my_paths = run_pred_recurse_for_source(K, 7)
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Execution time of mine: {execution_time:.6f} seconds")

start_time = time.perf_counter()

their_paths = []
for node in K.nodes:
    their_paths = their_paths +  [p for p in nx.all_shortest_paths(K, 7, node)] 

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Execution time of theirs: {execution_time:.6f} seconds")

# Yay mine is at least twice as fast
# And doesn't require try/catch

In [ ]:
#
# Now we actually calculate betweeness centrality
# 
from networkx import NetworkXNoPath

def compute_betweeness(network, print_statements=False):
    """
    This function calculates the inbetweeness for each node.

    It calculates inbetweeness of each node as the percentage of 
    shortest paths it falls on between any other two nodes divided
    by the total number of shortest paths between those nodes, summed for
    all nodes. 

    Basically, it loops through all the nodes and gets all the shortest paths 
    from that node to every other node (using my speedier function!). Then it loops through
    all the nodes again starting at itself (so that we don't calculate all the short paths
    twice, i.e. between Node A and Node C and Node C and Node A) and gets the subset
    of shortest paths. For every node in those paths, we add 1/(# of paths) to that node's
    betweeness score, which is stored in a dictionary.


    @param network (object): The network we are looking at
    @param print_statements (boolean): whether it prints obnoxious quantities of print statements

    @return dictionary of the betweeness scores for every single node 
    """

    # make sure it is a networkX object
    if type(network) != nx.Graph and type(network) != nx.DiGraph:
        raise Exception("Network must be NetworkX graph.")
    
    # Check to make sure graph has nodes
    if len(list(network.nodes)) < 1:
        raise Exception("Network must have nodes.")

    

    # First, get all the shortest paths and store in a dict 
    running_betweenness = dict()
    for node_x in network.nodes:
        if print_statements:
            print(f"\n\nStarting node x {node_x}...")
        # get all shortest paths
        all_node_paths = run_pred_recurse_for_source(network, node_x)
        if running_betweenness.get(node_x) == None:
            running_betweenness[node_x] = 0
        for i in range(int(node_x), len(network.nodes)):
            node_y = list(network.nodes)[i]
            if print_statements:
                print(f"node y : {node_y}")
            shortest_paths = [my_p for my_p in all_node_paths if my_p[-1] == node_y]
            if print_statements:
                print(f"shortest_paths: {shortest_paths}")
            for shorty in shortest_paths:
                shorty_innards = shorty[1:-1] # doesn't count if its an endpoint
                for el in shorty_innards:
                    current_val = running_betweenness.get(el, 0)
                    running_betweenness[el] = current_val + (1 / len(shortest_paths))
                    if print_statements:
                        print(f"Old val for {el}: {current_val}")
                        print(f"New val: {running_betweenness[el]}")
    if print_statements:
        print(f"Running betweeness: {running_betweenness}")
    return dict(sorted(running_betweenness.items()))
    

In [ ]:
import pprint
import time

# measuring runtime bc I'm psychotic maybe 
start_time = time.perf_counter()

btwn_dict = compute_betweeness(G, print_statements=False)

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.6f} seconds")

pprint.pprint(btwn_dict)

In [ ]:
# Measuring it against theirs
start_time = time.perf_counter()

their_dict = nx.betweenness_centrality(K, normalized=False)

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.6f} seconds")
pprint.pprint(their_dict)

# Theirs is obviously enormously faster

### Question 3b

In [ ]:
# Function above updated

# Test nodeless network
d = nx.Graph()
compute_betweeness(d)
 # Good, an exception

In [ ]:
# test non-Network object
s = object
compute_betweeness(s)
 # Good, an exception

### Question 3c

In [ ]:
# No links
no_links = nx.Graph()
no_links.add_nodes_from([1, 2, 3, 4, 5]) 
plt.figure(figsize=(4, 4))
nx.draw(no_links, with_labels=True, node_color='lightblue', edge_color='gray', node_size=800, font_size=10)
plt.title("No Link Graph")
plt.show()

In [ ]:
# Mine
print(compute_betweeness(no_links))
# Theirs 
print(nx.betweenness_centrality(no_links, normalized=False))

In [ ]:
# Two connected components
two_connected = nx.Graph()

two_connected.add_nodes_from([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]) 
two_connected.add_edges_from([(1,2), (2,3), (4,5), (3,6), (1,3), (4,9), (4,8)])
plt.figure(figsize=(5, 5))
nx.draw(two_connected, with_labels=True, node_color='lightblue', edge_color='gray', node_size=200, font_size=10)
plt.title("Separate Connected Parts")
plt.show()

In [ ]:
# Mine
print(compute_betweeness(two_connected))
# Theirs
print(nx.betweenness_centrality(two_connected, normalized=False))

print(compute_betweeness(two_connected) == nx.betweenness_centrality(two_connected, normalized=False))

In [ ]:
# More complex graph
import random

mister_complex = nx.Graph()

mister_complex.add_nodes_from(range(0,300)) 
for i in range(0,250):
    node_one = random.randint(0,300)
    node_two = random.randint(0,300)
    if node_one == node_two:
        continue
    mister_complex.add_edge(node_one, node_two)
plt.figure(figsize=(5, 5))
nx.draw(mister_complex, with_labels=True, node_color='lightblue', edge_color='gray', node_size=100, font_size=6)
plt.title("Mr. Complex")
plt.show()

In [ ]:
# Mine
print(compute_betweeness(mister_complex))
# Theirs
print(nx.betweenness_centrality(mister_complex, normalized=False))

print(compute_betweeness(mister_complex) == nx.betweenness_centrality(mister_complex, normalized=False)) 
# ruh roh, not the same! 

In [ ]:
# Who is the culprit?
my_results = compute_betweeness(mister_complex)
their_results = nx.betweenness_centrality(mister_complex, normalized=False)
for key, value in my_results.items():
    if value != their_results[key]:
        print("Found a bad one!")
        print(f"My results: {value} and theirs: {their_results[key]}")

# Looks like its that old crook, the ROUNDING ERROR!! 

In [ ]:
# Let's nab him 
my_results_rounded = {key: round(val, 2) for key, val in my_results.items()}
their_results_rounded = {key: round(val, 2) for key, val in their_results.items()}
print(my_results_rounded == their_results_rounded) 

# Got him!!!!!!

### Question 3d

In [ ]:
def test_betweenness_func(btwn_func):
    """
    This function tests a betweeness function.

    It tests both edgecases and correct cases, including:
    - Non-graph object 
    - Nodeless graph
    - Edgeless graph
    - Graph with one edge
    - Fully connected graph
    - Graph with multiple components
    - Complex random graph

    @param btwn_func (function): The function we want to test

    @return bool, error_message: whether or not the test suite succeeded, and error messages if it did not
    """

    # Non_graph object
    obj = object
    try:
        compute_betweeness(obj)
        return False, "Failed to throw exception when random python object was passed."
    except Exception as e:
        print(f"The following exception was properly thrown for random python object: {e}")

    # Test nodeless network
    d = nx.Graph()
    try:
        compute_betweeness(d)
        test_success = False
        return test_success, "Failed to throw exception when nodeless graph was passed."
    except Exception as e:
        print(f"The following exception was properly thrown for nodeless graph: {e}")

    # Edgeless graph
    edgeless =  nx.Graph()
    edgeless.add_nodes_from([1, 2, 3, 4, 5]) 

    if compute_betweeness(edgeless) != nx.betweenness_centrality(edgeless, normalized=False):
        return False, "Edgeless graph is not equivalent to NetworkX's edgeless graph."
    else:
        print("Passed no-edge network test.")
    
    # Graph with one edge
    one_edge =  nx.Graph()
    one_edge.add_nodes_from([1, 2, 3, 4, 5]) 
    one_edge.add_edge(3,5)
    
    if compute_betweeness(one_edge) != nx.betweenness_centrality(one_edge, normalized=False):
        return False, "One-edge graph is not equivalent to NetworkX's one-edge graph."
    else:
        print("Passed one-edge network test.")

    # Fully connected graph
    connected =  nx.Graph()
    connected.add_nodes_from([1, 2, 3, 4, 5]) 
    connected.add_edges_from([(1,2), (1,3), (1,4), (1,5), (2,3), (2,4), (2,5), (3,4), (3,5), (4,5)])
    
    if compute_betweeness(connected) != nx.betweenness_centrality(connected, normalized=False):
        return False, "Fully connected graph is not equivalent to NetworkX's fully connected graph."
    else:
        print("Passed fully connected network test.")

    # Graph with multiple components
    two_connected = nx.Graph()
    two_connected.add_nodes_from([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]) 
    two_connected.add_edges_from([(1,2), (2,3), (4,5), (3,6), (1,3), (4,9), (4,8)])

    if compute_betweeness(connected) != nx.betweenness_centrality(connected, normalized=False):
        return False, "Multicomponent graph is not equivalent to NetworkX's multicomponent graph."
    else:
        print("Passed multicomponent network test.")

    # Complex random graph
    mister_complex = nx.Graph()

    mister_complex.add_nodes_from(range(0,300)) 
    for i in range(0,250):
        node_one = random.randint(0,300)
        node_two = random.randint(0,300)
        if node_one == node_two:
            continue
        mister_complex.add_edge(node_one, node_two)
        
    our_results = compute_betweeness(mister_complex)
    their_results = nx.betweenness_centrality(mister_complex, normalized=False)

    my_results_rounded = {key: round(val, 2) for key, val in our_results.items()}
    their_results_rounded = {key: round(val, 2) for key, val in their_results.items()}

    if my_results_rounded != their_results_rounded:
        return False, "Complex graph is not equivalent to NetworkX's complex graph. Yes, we checked rounding error."
    else:
        print("Passed complex network test.")

    return True, "All tests passed! Great job."


In [ ]:
res, message = test_betweenness_func(compute_betweeness)
print(f"Result is: {res}: {message}")

## Question 4

In [ ]:
import requests 
from bs4 import BeautifulSoup
import re

### Question 4a

In [ ]:
def get_all_NE_depts():
    """
    This function queries the Northeastern University department listing page
    to get all of the departement abbreviations.

    @param None

    @return list: A list of all department abbreviations, e.g. ["PPUA"]
    """
        # First, get all department abbreviations from the catalog page
    catalog_res = requests.get('https://catalog.northeastern.edu/course-descriptions/')
    catalog_html = catalog_res.text
    soup = BeautifulSoup(catalog_html, 'html.parser')  # name the parser, or BeautifulSoup guesses and warns
    soup_alpha_div = soup.find('div', id="atozindex")
    department_hrefs = []
    for ul in soup_alpha_div.find_all('ul'):
        for li in ul.find_all('li'):
            href = li.a.get('href')
            dept_abbr = href.split('/')[-2]
            department_hrefs.append(dept_abbr)
    return sorted(department_hrefs)

def get_NE_course_titles_and_descriptions(dept_abbr):
    """
    This function queries the Northeastern University course catalog for a specific 
    department and returns a dictionary of information about each course, 
    including its prerequisites.

    @param dept_abbr (string): The abbreviation of the department we are querying, e.g. "PPUA" for Public Policy

    @return dict: A dictionary containing each course number as key, and a nested dictionary with 
    title, description, and prerequisites as values.
    """

    url = 'https://catalog.northeastern.edu/course-descriptions/' + dept_abbr.lower() + "/"
    dept_html = requests.get(url).text

    total_soup = BeautifulSoup(dept_html, 'html.parser')  # name the parser, or BeautifulSoup guesses and warns
    all_courses = total_soup.find('div', id="textcontainer")
    title_dict = {}
    for i, course_block in enumerate(all_courses.find_all('div', class_='courseblock')):
        title = course_block.find('p', class_='courseblocktitle')
        title = title.text
        number = title.split('.')[0]
        title = title.split('.')[1]
        title = title.strip()
        number = number.replace("\xa0", "").strip()
        content = course_block.find('p', class_='cb_desc')
        pre_reqs = get_course_prerequisites(course_block)
        title_dict[number] = {"Title": title, "Content": content.text, "Prerequisites": pre_reqs}
    return title_dict


def get_course_prerequisites(course_block):
    """
    This function parses the course block of text for prerequisites.

    @param course_block (BS object): The HTML element containing the course information

    @return dict: A unique list of prerequisites for the course (as abbreviations).
    """
    other_info = course_block.find('p', class_='courseblockextra')
    if other_info:
        other_info = other_info.text
        if("Prerequisite" in other_info):
            course_pattern = r"[A-Z]{2,4}\s[\d]{4}"
            course_abbrs = re.findall(course_pattern, other_info)
            for i,abbr in enumerate(course_abbrs):
                course_abbrs[i] = abbr.replace("\xa0", "").replace(" ", "")
            final_abbrs = list(set(course_abbrs))
            return final_abbrs

In [ ]:
# First, get all departments
NE_depts = get_all_NE_depts()

# Next, get all course info for all departments
NE_courses = []

for department in NE_depts:
    dept_courses = get_NE_course_titles_and_descriptions(department)
    NE_courses.append(dept_courses)
    print(f"Appending {department} courses to NE_courses. Total courses so far: {len(NE_courses)}")
    print(dept_courses)

### Question 4b

In [ ]:

course_graph = nx.DiGraph()

for dept_courses in NE_courses:
    for course_num, course_info in dept_courses.items():
        course_preqs = course_info["Prerequisites"]
        if not course_graph.has_node(course_num):
            course_graph.add_node(course_num, department=course_num, title=course_info["Title"], content=course_info["Content"])

        if course_preqs == None:
            continue
        for preq in course_preqs:
            if not course_graph.has_node(preq):
                course_graph.add_node(preq)
            course_graph.add_edge(preq, course_num)

print(course_graph.number_of_nodes())
print(course_graph.number_of_edges())


In [ ]:

import matplotlib.pyplot as plt

# Visualize network
pos = nx.spring_layout(course_graph, k=0.9, iterations=50, seed=42)

plt.figure(figsize=(6, 6))
nx.draw_networkx(
    course_graph, 
    pos=pos, 
    arrows=True,             
    with_labels=False,        
    node_size=2,          
    node_color="skyblue",   
    arrowsize=2,
    width=0.4,              # thinner edges so they don't visually merge
    alpha=.8
)

plt.title("Northeastern University Course Prerequisite Network", fontsize=14)
plt.axis("off")
plt.show()

### 4c

In [ ]:
avg_preqs = {}
for dept_courses in NE_courses:
   # print(dept_courses)
    pre_requ_count = 0
    if len(dept_courses) == 0:
        continue
    dept = list(dept_courses.keys())[0].split()[0]
    dept_name = re.sub(r'\d+', '', dept)
    for course_num, course_info in dept_courses.items():
        course_preqs = course_info["Prerequisites"]
        if course_preqs == None:
            continue
        for preq in course_preqs:
            pre_requ_count += 1
    avg_preqs[dept_name] = pre_requ_count / len(dept_courses)

print(avg_preqs)
print(f"Maximunum avg prereq: {max(avg_preqs, key=avg_preqs.get)} with {max(avg_preqs.values())} average prereqs per course")

### Question 3d

In [ ]:
print(nx.is_directed_acyclic_graph(course_graph))

cycle_edges = nx.simple_cycles(course_graph)
print("Edges in the cycle:")
for edge in cycle_edges:
    print(edge)

## Question 5

### Question 5a

In [ ]:
dublin_network = nx.read_edgelist("data/ia-contacts_dublin/ia-contacts_dublin.edges", delimiter=",", data=[("timestamp", int)])

print(dublin_network.number_of_nodes())
print(dublin_network.number_of_edges())

hist = nx.degree_histogram(dublin_network)
plt.figure(figsize=(8, 5))
plt.bar(range(len(hist)), hist, width=0.8, color="blue")

plt.title("Original Network Degree Distribution")
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.show()

### Question 5b

In [ ]:
dublin_network_r = dublin_network.copy()
dublin_network_rand = nx.double_edge_swap(dublin_network_r, nswap=445170, max_tries=1005170)

In [ ]:
hist_rand = nx.degree_histogram(dublin_network_rand)


plt.figure(figsize=(8, 5))
plt.bar(range(len(hist_rand)), hist_rand, width=0.8, color="blue")

plt.title("Random Network Degree Distribution")
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.show()

# Degree distributions appear equal

#### Question 5b

In [ ]:
# Get betweeness centrality, used below
betweenness = nx.betweenness_centrality(dublin_network)
sorted_betweenness = np.sort(list(betweenness.values()))

In [ ]:
# Get edge betweeness centrality, used below
edge_betweenness = nx.edge_betweenness_centrality(dublin_network)
sorted_edge_betweenness = np.sort(list(edge_betweenness.values()))

In [ ]:
# Three distributions: clustering, node betweeness, and edge betweeness

fig, ax = plt.subplots(1,3,figsize=(14,5),dpi=200)

# Clustering (cumulative so it doesn't look awful)
clusters = nx.clustering(dublin_network)
sorted_clusters = np.sort(list(clusters.values()))
cdf = 1 - np.arange(len(sorted_clusters)) / len(sorted_clusters)

#ax[0].plot(sorted_clusters, cdf, color='cyan', alpha=0.8)
ax[0].loglog(sorted_clusters, cdf, color='cyan', alpha=0.8)
ax[0].set_xlabel(r"$C(v)$",fontsize='medium')
ax[0].set_ylabel(r"$P(C(v))\geq x$",fontsize='medium')
ax[0].grid(linewidth=1, color="#D4D4D4", alpha=0.2, linestyle='-')
ax[0].set_title('Dublin Contacts Clustering Distribution (linear)',fontsize='medium')

# Betweeness (in the above cell because it is so slow)
sorted_betweenness_final = [x for x in sorted_betweenness if x != 0]
cdf_btwn = 1 - np.arange(len(sorted_betweenness_final)) / len(sorted_betweenness_final)

ax[1].loglog(sorted_betweenness_final, cdf_btwn, color='blue', alpha=0.8)
ax[1].set_xlabel(r"$b(v)$",fontsize='medium')
ax[1].set_ylabel(r"$P(b(v))\geq x$",fontsize='medium')
ax[1].grid(linewidth=1, color="#D4D4D4", alpha=0.2, linestyle='-')
ax[1].set_title('Dublin Contacts Betweenness Distribution (log-log)',fontsize='medium')

# Now edge betweeness
sorted_edge_betweenness_final = [x for x in sorted_edge_betweenness if x != 0]
edge_cdf_btwn = 1 - np.arange(len(sorted_edge_betweenness_final)) / len(sorted_edge_betweenness_final)

ax[2].loglog(sorted_edge_betweenness_final, edge_cdf_btwn, color='green', alpha=0.8)
ax[2].set_xlabel(r"$b(e)$",fontsize='medium')
ax[2].set_ylabel(r"$P(b(e))\geq x$",fontsize='medium')
ax[2].grid(linewidth=1, color="#D4D4D4", alpha=0.2, linestyle='-')
ax[2].set_title('Dublin Contacts Edge Betweenness Distribution (log-log)',fontsize='medium')

plt.savefig("outputs/dublin_graph.png")
plt.show()

#### Question 5b ii

In [ ]:
# Get betweeness centrality, used below for random network
betweenness_rand = nx.betweenness_centrality(dublin_network_rand)
sorted_betweenness_rand = np.sort(list(betweenness_rand.values()))

In [ ]:
# Get edge betweeness centrality, used below for random network
edge_betweenness_rand = nx.edge_betweenness_centrality(dublin_network_rand)
sorted_edge_betweenness_rand = np.sort(list(edge_betweenness_rand.values()))

In [ ]:
# Now the random network

# Three distributions: clustering, node betweeness, and edge betweeness

fig, ax = plt.subplots(1,3,figsize=(14,5),dpi=200)

# Clustering (cumulative so it doesn't look awful)
clusters_rand = nx.clustering(dublin_network_rand)
sorted_clusters_rand = np.sort(list(clusters_rand.values()))
cdf_rand = 1 - np.arange(len(sorted_clusters_rand)) / len(sorted_clusters_rand)

#ax[0].plot(sorted_clusters_rand, cdf, color='cyan', alpha=0.8)
ax[0].loglog(sorted_clusters_rand, cdf_rand, color='cyan', alpha=0.8)
ax[0].set_xlabel(r"$C(v)$",fontsize='medium')
ax[0].set_ylabel(r"$P(C(v))\geq x$",fontsize='medium')
ax[0].grid(linewidth=1, color="#D4D4D4", alpha=0.2, linestyle='-')
ax[0].set_title('Random Clustering Distribution (linear)',fontsize='medium')

# Betweeness (in the above cell because it is so slow)
sorted_betweenness_rand = [x for x in sorted_betweenness_rand if x != 0]
cdf_btwn = 1 - np.arange(len(sorted_betweenness_rand)) / len(sorted_betweenness_rand)

ax[1].loglog(sorted_betweenness_rand, cdf_btwn, color='blue', alpha=0.8)
ax[1].set_xlabel(r"$b(v)$",fontsize='medium')
ax[1].set_ylabel(r"$P(b(v))\geq x$",fontsize='medium')
ax[1].grid(linewidth=1, color="#D4D4D4", alpha=0.2, linestyle='-')
ax[1].set_title('Random Betweenness Distribution (log-log)',fontsize='medium')

# Now edge betweeness
sorted_edge_betweenness_rand = [x for x in sorted_edge_betweenness_rand if x != 0]
edge_cdf_btwn = 1 - np.arange(len(sorted_edge_betweenness_rand)) / len(sorted_edge_betweenness_rand)

ax[2].loglog(sorted_edge_betweenness_rand, edge_cdf_btwn, color='green', alpha=0.8)
ax[2].set_xlabel(r"$b(e)$",fontsize='medium')
ax[2].set_ylabel(r"$P(b(e))\geq x$",fontsize='medium')
ax[2].grid(linewidth=1, color="#D4D4D4", alpha=0.2, linestyle='-')
ax[2].set_title('Random Edge Betweenness Distribution (log-log)',fontsize='medium')

plt.savefig("random_graph.png")
plt.show()


In [ ]:
# Just to see actual data
fig, ax = plt.subplots(1, 3, figsize=(14, 4))

# Clustering
clusters_rand = nx.clustering(dublin_network)
ax[0].hist(list(clusters_rand.values()), bins=30, color='cyan', edgecolor='black')
ax[0].set_xlabel("C(v)")
ax[0].set_ylabel("count")
ax[0].set_title("Clustering")

# Node betweenness
ax[1].hist(list(sorted_betweenness_rand.values()), bins=30, color='blue', edgecolor='black')
ax[1].set_xlabel("b(v)")
ax[1].set_ylabel("count")
ax[1].set_title("Node Betweenness")

# Edge betweenness
ax[2].hist(list(sorted_edge_betweenness_rand.values()), bins=30, color='green', edgecolor='black')
ax[2].set_xlabel("b(e)")
ax[2].set_ylabel("count")
ax[2].set_title("Edge Betweenness")

plt.tight_layout()
plt.show()